In [2]:
# ============================================================
# Add MACE model MRE columns to Statistical_errors.csv
#
# Adds:
#   MACE-POLAR-1-S
#   MACE-POLAR-1-M
#   MACE-POLAR-1-L
#   MACE-OMOL-0-100M
#   MACE-OMOL-0-4M
#   MACE-MP-0b3
#
# Missing datatype/model combinations remain BLANK.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path.cwd().parent if Path.cwd().name == "Analysis" else Path.cwd()
A = ROOT / "Analysis"
I = ROOT / "Info"

STAT_FILE = A / "Statistical_errors.csv"
if not STAT_FILE.exists():
    STAT_FILE = ROOT / "Statistical_errors.csv"

# ------------------------------------------------------------
# Load Statistical_errors.csv
# IMPORTANT: there is NO "Datatype" column
# ------------------------------------------------------------

stat = pd.read_csv(STAT_FILE)

# First column contains:
# Mean, Mean Barrier Height, Mean Electric field, ...
stat = stat.rename(columns={stat.columns[0]: "Statistic"})
stat = stat.set_index("Statistic")

# ------------------------------------------------------------
# Output model names
# ------------------------------------------------------------

MODELS = [
    "MACE-POLAR-1-S",
    "MACE-POLAR-1-M",
    "MACE-POLAR-1-L",
    "MACE-OMOL-0-100M",
    "MACE-OMOL-0-4M",
    "MACE-MP-0b3"
]

for model in MODELS:
    stat[model] = np.nan

# ------------------------------------------------------------
# Canonical column aliases used by your different CSV files
# ------------------------------------------------------------

ALIASES = {
    "MACE-POLAR-1-S":
        ["MACE-POLAR-1-S", "mace-polar-1-s"],

    "MACE-POLAR-1-M":
        ["MACE-POLAR-1-M", "mace-polar-1-m"],

    "MACE-POLAR-1-L":
        ["MACE-POLAR-1-L", "mace-polar-1-l"],

    "MACE-OMOL-0-100M":
        ["MACE-OMOL-0-100M"],

    "MACE-OMOL-0-4M":
        ["MACE-OMOL-0-4M"],

    "MACE-MP-0b3":
        ["MACE-MP-0b3", "MACE-MP-0b3-medium"]
}

# ------------------------------------------------------------
# Standard GSCDB metric used for MRE normalization
#
# MRE_dataset = model metric / GSCDB standard metric
# ------------------------------------------------------------

std = pd.read_csv(I / "Standard_errors.csv").set_index("Dataset")["Metric"]

info = pd.read_csv(I / "Datasets.csv", index_col=0)
info.index = info.index.astype(str)

# ------------------------------------------------------------
# Master per-dataset MRE table
# ------------------------------------------------------------

mre = pd.DataFrame(
    index=std.index.astype(str),
    columns=MODELS,
    dtype=float
)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def find_file(name):
    """Look in ROOT and Analysis/."""
    for p in [ROOT / name, A / name]:
        if p.exists():
            return p
    return None


def get_column(df, model):
    """Find the actual CSV column corresponding to a model."""
    for name in ALIASES[model]:
        if name in df.columns:
            return name
    return None


def dataset_index(df):
    """Convert a per-dataset CSV to Dataset-indexed form."""
    if "Dataset" in df.columns:
        return df.set_index("Dataset")

    first = df.columns[0]

    if str(first).startswith("Unnamed"):
        return df.set_index(first)

    return df


def add_direct_mre(filename):
    """
    Add values from files that already contain
    per-dataset relative metric / MRE.
    """
    p = find_file(filename)

    if p is None:
        return

    d = dataset_index(pd.read_csv(p))
    d.index = d.index.astype(str)

    for model in MODELS:

        col = get_column(d, model)

        if col is not None:

            values = pd.to_numeric(
                d[col],
                errors="coerce"
            )

            common = mre.index.intersection(values.index)

            # Do not overwrite an already valid value
            old = mre.loc[common, model]

            mre.loc[common, model] = old.fillna(
                values.reindex(common)
            )


def add_mae_file(filename):
    """
    Convert dataset-level MAE to MRE using Standard_errors.csv:

            MRE = MAE / StandardMetric
    """
    p = find_file(filename)

    if p is None:
        return

    d = dataset_index(pd.read_csv(p))
    d.index = d.index.astype(str)

    common = mre.index.intersection(d.index)

    for model in MODELS:

        col = get_column(d, model)

        if col is None:
            continue

        mae = pd.to_numeric(
            d.loc[common, col],
            errors="coerce"
        )

        values = mae / std.reindex(common)

        # Direct MRE files have priority
        old = mre.loc[common, model]

        mre.loc[common, model] = old.fillna(values)


# ============================================================
# 1. DIRECT per-dataset MRE files
# ============================================================

# MACE-POLAR — Barrier Height
add_direct_mre(
    "Relative_metric_per_set_MACE_POLAR_Datatype_Barrier_Height_ALL3.csv"
)

# MACE-POLAR — Isomerization
add_direct_mre(
    "Relative_metric_per_set_MACE_POLAR_Datatype_Isomerization_ALL3.csv"
)

# MACE-POLAR — Noncovalent
# Important because O24/O24x4 use their special metric treatment.
add_direct_mre(
    "Errors_per_set_MRE_MACE_POLAR_Datatype_Noncovalent_ALL3.csv"
)

# MACE-POLAR — Transition Metal
# Important because TMD10/MOR13/TMB11 use the special metric treatment.
add_direct_mre(
    "Relative_metric_per_set_MACE_POLAR_Datatype_Transition_Metal_ALL3.csv"
)

# MACE-MP-0b3
add_direct_mre(
    "Relative_metric_per_set_MACE_MP_0b3_Datatype_Barrier_Height.csv"
)

add_direct_mre(
    "Relative_metric_per_set_MACE_MP_0b3_Datatype_Intramolecular_Noncovalent.csv"
)

add_direct_mre(
    "Relative_metric_per_set_MACE_MP_0b3_Datatype_Frequency.csv"
)

# MACE-OMOL-0
add_direct_mre(
    "Relative_metric_per_set_MACE_OMOL_Datatype_Intramolecular_Noncovalent_ALL2.csv"
)

add_direct_mre(
    "Relative_metric_per_set_MACE_OMOL_Datatype_Frequency_ALL2.csv"
)

# ============================================================
# 2. Combined MAE files
#
# Used as fallback for model/datatype combinations for which
# direct per-set MRE files are not present.
# ============================================================

add_mae_file(
    "Barrier Height Datatype_MAE Results_MACE models.csv"
)

add_mae_file(
    "Intramolecular Noncovalent Datatype_MAE Results_MACE models.csv"
)

add_mae_file(
    "Thermochemistry Datatype_MAE Results_MACE models.csv"
)

# ============================================================
# 3. Frequency — V30
# ============================================================

freq_file = find_file(
    "Frequency_Datatype_MAE_Results_All_MACE_models.csv"
)

if freq_file is not None:

    f = pd.read_csv(freq_file)

    # Your Frequency file is LONG format:
    # Dataset, Datatype, Model, MAE_cm^-1
    if {"Dataset", "Model", "MAE_cm^-1"}.issubset(f.columns):

        for model in MODELS:

            aliases = ALIASES[model]

            q = f[
                f["Model"].isin(aliases)
            ]

            for _, row in q.iterrows():

                dataset = str(row["Dataset"])

                if dataset in mre.index:

                    mre.loc[dataset, model] = (
                        float(row["MAE_cm^-1"])
                        / std.loc[dataset]
                    )

# ============================================================
# 4. Frequency — MACE-POLAR direct MRE
# ============================================================

v30_file = find_file(
    "V30_ALL3_metrics_RMSE_MAE_MRE.csv"
)

if v30_file is not None:

    v = pd.read_csv(v30_file)

    if {"Model", "MRE"}.issubset(v.columns):

        for model in [
            "MACE-POLAR-1-S",
            "MACE-POLAR-1-M",
            "MACE-POLAR-1-L"
        ]:

            alias = ALIASES[model]

            q = v[
                v["Model"].isin(alias)
            ]

            if len(q):

                mre.loc["V30", model] = float(
                    q["MRE"].iloc[0]
                )

# ============================================================
# 5. ELECTRIC FIELD
#
# Dip146, HR46, Pol130, T144 and OEEF.
#
# Only MACE-POLAR values are currently available in these
# files, so OMOL/MP columns naturally remain blank.
# ============================================================

ELECTRIC_FILES = [
    "Dip146_ALL3_CORRECTED_metrics.csv",
    "HR46_ALL3_FIXED_V2_metrics.csv",
    "Pol130_ALL3_metrics.csv",
    "T144_ALL3_metrics.csv",
    "OEEF_ALL3_metrics.csv"
]

for filename in ELECTRIC_FILES:

    p = find_file(filename)

    if p is None:
        continue

    e = pd.read_csv(p)

    if not {"Dataset", "Model", "MAE"}.issubset(e.columns):
        continue

    for _, row in e.iterrows():

        dataset = str(row["Dataset"])

        if dataset not in std.index:
            continue

        for model in [
            "MACE-POLAR-1-S",
            "MACE-POLAR-1-M",
            "MACE-POLAR-1-L"
        ]:

            if row["Model"] in ALIASES[model]:

                mre.loc[dataset, model] = (
                    float(row["MAE"])
                    / std.loc[dataset]
                )

# ============================================================
# 6. Fill DATATYPE rows of Statistical_errors.csv
# ============================================================

# Normalise capitalisation because the CSV has
# "Mean Electric field" but Datasets.csv may use "Electric Field".
datatype_lookup = {}

for datatype in info["Datatype"].dropna().unique():

    datasets = info.index[
        info["Datatype"].eq(datatype)
    ].astype(str)

    datatype_lookup[
        str(datatype).strip().lower()
    ] = list(datasets)


for row in stat.index:

    if row == "Mean":
        continue

    if not str(row).startswith("Mean "):
        continue

    datatype_name = str(row)[5:].strip()
    key = datatype_name.lower()

    datasets = datatype_lookup.get(key, [])

    # case-insensitive fallback
    if not datasets:
        for k, v in datatype_lookup.items():
            if k.lower() == key.lower():
                datasets = v
                break

    for model in MODELS:

        values = (
            mre.reindex(datasets)[model]
            .dropna()
        )

        if len(values):
            stat.loc[row, model] = values.mean()

# ============================================================
# 7. Overall "Mean"
#
# Same idea as the original analyze.ipynb:
# mean of PER-DATASET MRE values, not mean of datatype means.
#
# Missing model calculations are ignored.
# ============================================================

for model in MODELS:

    values = mre[model].dropna()

    if len(values):
        stat.loc["Mean", model] = values.mean()

# ============================================================
# 8. Save
# ============================================================

OUT = A / "Statistical_errors_with_ALL_MACE_models.csv"

stat.to_csv(
    OUT,
    index=True,
    na_rep=""       # missing results stay BLANK
)

# Display
display(stat.round(4))

print(f"\nSaved:\n{OUT}")

,wB97M2,wB97M-V,CF22D,wB97X-V,revDSD-PBEP86-D4,M052X,M062X,M08HX,MN15,r2SCAN0,...,CAMB3LYP,SOGGA11X,BMK,B3LYP,MACE-POLAR-1-S,MACE-POLAR-1-M,MACE-POLAR-1-L,MACE-OMOL-0-100M,MACE-OMOL-0-4M,MACE-MP-0b3
Statistic,,,,,,,,,,,,,,,,,,,,,
Mean,0.9485,1.0755,1.2493,1.3129,1.5220,1.9415,1.9710,1.9935,2.0099,2.8117,...,3.6480,4.1149,4.5236,5.6912,28.4674,7.6859,51.3596,11.3041,954.5606,348.8192
Mean Barrier Height,0.6682,1.0616,1.1170,1.5680,1.2165,1.9802,1.5384,1.2538,1.3679,2.0759,...,2.2616,1.5385,1.6316,3.2929,5.8012,2.0176,1.5162,1.4996,2.5087,12.4864
Mean Electric field,1.0389,1.8072,1.9816,1.0409,1.8874,1.0623,1.0209,1.2059,1.5372,1.1161,...,1.2987,1.0893,1.0451,2.0689,331.6308,92.1503,95.6997,NaN,NaN,NaN
Mean Frequency,0.8761,0.9247,2.8605,1.1907,0.3264,1.3229,1.0136,1.2513,1.7629,1.6660,...,1.0621,1.3628,1.0997,1.0608,1.7823,1.5142,1.5246,1.5206,1.6783,6.4853
Mean Intramolecular Noncovalent,1.0213,1.0708,1.2542,0.9299,1.0032,1.7075,1.9743,2.9647,3.7178,3.4250,...,5.7560,4.4651,2.8606,8.0697,2.9915,1.6892,1.5525,1.4313,1.5603,20.6503
Mean Isomerization,0.5555,1.3026,1.2206,1.9040,3.0664,1.7614,1.6087,1.5947,1.7685,2.7629,...,3.6031,3.2523,2.3406,4.9207,1.9246,1.7132,1.4626,NaN,NaN,NaN
Mean Noncovalent,1.0975,0.8907,1.4086,0.9786,1.2321,2.6458,2.6229,2.5427,2.3195,3.6078,...,5.3557,7.4330,9.0509,9.1630,15.8079,5.7918,136.1799,NaN,NaN,NaN
Mean Thermochemistry,0.9246,1.1100,1.0515,1.5674,1.6537,1.4631,1.5781,1.5719,1.7851,2.5259,...,2.5052,2.3918,2.5973,3.6921,30.5687,5.3693,5.9006,16.5638,1488.8900,526.9591
Mean Transition Metal,1.0519,1.1877,1.0392,1.2681,1.3530,1.7444,2.2826,2.2230,1.1039,1.6569,...,1.8504,2.0348,1.9895,2.1287,4.5821,2.9831,2.7368,NaN,NaN,NaN



Saved:
c:\Users\91988\Documents\GitHub\Accurate-Prediction-of-Molecular-Properties-with-Polarisable-MACE\Analysis\Statistical_errors_with_ALL_MACE_models.csv
